In [1]:
from json import load
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from scipy.ndimage import label, binary_dilation

from copy import deepcopy

from itertools import permutations

from collections import deque

import heapq

DIRPATH = '../maps/example_maps_odom_v6_003_waffle_counter_clockwise'
MAP_001 = 'map_odom_20250505_213432_972816.json'

VAL_UNKNOWN = -1
VAL_FREE = 0
VAL_OCCUPIED = 100
VAL_INACCESSIBLE = 101

VAL_CURR_POSITION = 200
VAL_NEXT_GOAL = 201

VAL_ESTIMATED_WALL = 250

colormap = {
    VAL_UNKNOWN: [0.5, 0.5, 0.5],  # Gray (unknown terrain)
    VAL_FREE: [1.0, 1.0, 1.0],  # White (blank space)
    VAL_OCCUPIED: [0.0, 0.0, 0.0],  # Black (walls)
    VAL_INACCESSIBLE: [0.0, 1.0, 0.0],  # Green (estimated walls)
    
    VAL_CURR_POSITION: [1.0, 0.0, 0.0],  # Red (current position)
    VAL_NEXT_GOAL: [0.0, 0.0, 1.0],  # Blue (next goal)
    VAL_ESTIMATED_WALL: [1.0, 0.5, 0.0],  # Orange (estimated walls)
}

RESOLUTION = 0.03  # meters per pixel

In [2]:
def read_data_map(_dir_path: str=DIRPATH, _map_path: str=MAP_001) -> tuple[np.ndarray[tuple[()], np.dtype], float, float]:
    """Reads map data from a JSON file and returns the grid, origin x, and origin y.
    Args:
        _dir_path (str, optional): Directory path to the map file. Defaults to DIRPATH.
        _map_path (str, optional): Map file name. Defaults to MAP_001.

    Returns:
        _type_: _description_
    """
    
    _data = dict()
    with open(_dir_path + '/' + _map_path, 'r+') as json_data:
        _data = load(json_data)
        json_data.close()
    
    try:
        _data = _data['map']
        print(_data)

        width = _data['info']['width']
        height = _data['info']['height']
        origin_x = _data['info']['origin']['position']['x']
        origin_y = _data['info']['origin']['position']['y']
        _data = _data['data']

        grid = np.array(_data).reshape((height, width))

        return grid, origin_x, origin_y

    except:
        print('No data found in the map file')
    
    return None, None, None

def read_data_odom(_dir_path: str=DIRPATH, _odom_path: str=MAP_001) -> tuple[int, int]:
    _data = dict()
    with open(_dir_path + '/' + _odom_path, 'r+') as json_data:
        _data = load(json_data)
        json_data.close()
    
    try:
        _data = _data['odom']
        print(_data)

        x = _data['pose']['pose']['position']['x']
        y = _data['pose']['pose']['position']['y']

        return x, y

    except:
        print('No data found in the odom file')
    
    return None, None
    

def print_plot_v2(
    grid: np.ndarray[tuple[()], np.dtype],
    title: str="Original map",
    filename: str="Not provided",
    occupied: int=-1,
    free: int=-1,
    inaccessible: int=-1,
    unknown: int=-1,
    explored_percent: float=-1.0,
    position_x: float=-1,
    position_y: float=-1) -> None:
    height, width = grid.shape
    colored_map = np.zeros((height, width, 3))

    for y in range(height):
        for x in range(width):
            color_code = grid[y, x]
            colored_map[y, x] = colormap[color_code]
            
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(colored_map, origin='upper')
    plt.title(title)
    plt.axis('off')
    
    legend_text = (f"Filename: {filename}\n"
                   f"Occupied: {occupied}\n"
                   f"Free: {free}\n"
                   f"Inaccessible: {inaccessible}\n"
                   f"Unknown: {unknown}\n"
                   f"Explored: {100*explored_percent:.8f}\n"
                   f"Position: {position_x:.4f}, {position_y:.4f}"
                   )
    
    plt.subplot(1, 2, 2)
    plt.axis('off')
    plt.text(0.1, 0.5, legend_text, fontsize=12, verticalalignment='center')
    
    plt.show()
    
    return

## Greedy Frontier

In [3]:
def find_frontiers(grid: np.ndarray) -> np.ndarray:
    unknown = grid == -1
    known = (grid == 0) | (grid == 100) | (grid == 200)
    frontiers = np.zeros_like(grid, dtype=bool)
    h, w = grid.shape
    for y in range(1, h-1):
        for x in range(1, w-1):
            if known[y, x] and -1 in grid[y-1:y+2, x-1:x+2]:
                frontiers[y, x] = True
    return frontiers

def a_star(grid: np.ndarray, start: tuple[int, int], goal: tuple[int, int]) -> list[tuple[int, int]]:
    h, w = grid.shape
    visited = set()
    pq = [(0, start)]
    came_from = {start: None}
    cost = {start: 0}
    while pq:
        _, current = heapq.heappop(pq)
        if current == goal:
            path = []
            while current:
                path.append(current)
                current = came_from[current]
            return path[::-1]
        if current in visited:
            continue
        visited.add(current)
        for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
            nx, ny = current[0] + dx, current[1] + dy
            if 0 <= nx < w and 0 <= ny < h and grid[ny, nx] == 0:
                next_cost = cost[current] + 1
                if (nx, ny) not in cost or next_cost < cost[(nx, ny)]:
                    cost[(nx, ny)] = next_cost
                    priority = next_cost
                    heapq.heappush(pq, (priority, (nx, ny)))
                    came_from[(nx, ny)] = current
    return []

def simulate_visibility(grid: np.ndarray, position: tuple[int, int], radius: int=45) -> np.ndarray:
    y, x = position
    yy, xx = np.ogrid[:grid.shape[0], :grid.shape[1]]
    mask = (yy - y)**2 + (xx - x)**2 <= radius**2
    visible = (grid == -1) & mask
    grid[visible] = 0  # Mark as seen
    return grid

def greedy_frontier_exploration(grid: np.ndarray, start: tuple[int, int]) -> list[tuple[int, int]]:
    new_grid = deepcopy(grid)
    path = []
    grids = [new_grid]
    current = start
    while True:
        frontiers = find_frontiers(grid)
        if not np.any(frontiers):
            break
        
        print(frontiers)
        frontier_indices = np.argwhere(frontiers)
        if len(frontier_indices) == 0:
            break
        fy, fx = frontier_indices[:, 0], frontier_indices[:, 1]
        dists = [np.linalg.norm([fx[i] - current[0], fy[i] - current[1]]) for i in range(len(fx))]
        nearest = (fx[np.argmin(dists)], fy[np.argmin(dists)])
        subpath = a_star(new_grid, current, nearest)
        if not subpath:
            break
        for p in subpath:
            new_grid = simulate_visibility(new_grid, (p[1], p[0]))
        path += subpath
        grids.append(new_grid)
        current = nearest
    return path, grids

## Ant colony

In [4]:
def ant_colony_exploration(grid, start, iterations=50, n_ants=10, alpha=1.0, beta=2.0, evaporation=0.5):
    frontiers = np.argwhere(find_frontiers(grid))
    pheromones = np.ones((grid.shape[0], grid.shape[1]))
    best_path = None
    best_length = np.inf

    def heuristic(p1, p2):
        return 1.0 / (np.linalg.norm(np.array(p1) - np.array(p2)) + 1e-6)

    for _ in range(iterations):
        all_paths = []
        for _ in range(n_ants):
            pos = start
            path = [pos]
            visited = set()
            while True:
                neighbors = []
                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nx, ny = pos[0] + dx, pos[1] + dy
                    if (0 <= nx < grid.shape[1] and 0 <= ny < grid.shape[0] and
                        grid[ny, nx] == 0 and (nx, ny) not in visited):
                        neighbors.append((nx, ny))
                if not neighbors:
                    break
                weights = [
                    (pheromones[ny, nx]**alpha) * (heuristic((nx, ny), frontiers[0][::-1])**beta)
                    for nx, ny in neighbors
                ]
                probs = weights / np.sum(weights)
                pos = neighbors[np.random.choice(len(neighbors), p=probs)]
                visited.add(pos)
                path.append(pos)
                if tuple(pos[::-1]) in map(tuple, frontiers):
                    break
            all_paths.append((path, len(path)))

        # Update pheromones
        pheromones *= (1 - evaporation)
        for path, length in all_paths:
            for x, y in path:
                pheromones[y, x] += 1.0 / length
            if length < best_length:
                best_length = length
                best_path = path

    for p in best_path:
        grid = simulate_visibility(grid, (p[1], p[0]))
    return best_path

## Clustering-Based Frontier Groupping

In [5]:
def clustering_frontier_exploration(grid, start, n_clusters=5):
    frontiers = np.argwhere(find_frontiers(grid))
    if len(frontiers) == 0:
        return []

    kmeans = KMeans(n_clusters=min(n_clusters, len(frontiers)))
    kmeans.fit(frontiers)
    centroids = kmeans.cluster_centers_.astype(int)

    path = []
    current = start
    for cx, cy in centroids:
        subpath = a_star(grid, current, (cx, cy))
        if not subpath:
            continue
        for p in subpath:
            grid = simulate_visibility(grid, (p[1], p[0]))
        path += subpath
        current = (cx, cy)
    return path

## Traveling Salesman Problem (TSP) Solver

In [6]:
def tsp_frontier_exploration(grid, start):
    frontiers = np.argwhere(find_frontiers(grid))
    points = [(x, y) for y, x in frontiers]
    if not points:
        return []

    def distance(p1, p2):
        return np.linalg.norm(np.array(p1) - np.array(p2))

    best_order = []
    best_cost = float('inf')

    for order in permutations(points):
        cost = distance(start, order[0]) + sum(distance(order[i], order[i+1]) for i in range(len(order)-1))
        if cost < best_cost:
            best_cost = cost
            best_order = order

    path = []
    current = start
    for pt in best_order:
        subpath = a_star(grid, current, pt)
        if not subpath:
            continue
        for p in subpath:
            grid = simulate_visibility(grid, (p[1], p[0]))
        path += subpath
        current = pt
    return path

In [7]:
def get_reachable_mask(grid: np.ndarray, position: tuple[int, int]) -> np.ndarray[tuple[()], np.dtype]:
    h, w = grid.shape
    visited = np.zeros_like(grid, dtype=bool)
    q = deque([position])
    visited[position[1], position[0]] = True
    
    while q:
        x, y = q.popleft()
        for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < w and 0 <= ny < h:
                if not visited[ny, nx] and grid[ny, nx] in [0, -1]:
                    visited[ny, nx] = True
                    q.append((nx, ny))

    return visited

def fill_outside_with_val_inaccessible(grid: np.ndarray, position: tuple[int, int]) -> np.ndarray[tuple[()], np.dtype]:
    reachable = get_reachable_mask(grid, position)
    grid[(grid == -1) & (~reachable)] = VAL_INACCESSIBLE
    
    return grid

def is_fully_enclosed(grid: np.ndarray, position: tuple[int, int]) -> bool:
    reachable = get_reachable_mask(grid, position)
    unknown_mask = (grid == -1)
    
    return not np.any(reachable & unknown_mask)

def fill_enclosed_unknowns_v2(grid: np.ndarray, position: tuple[int, int]) -> np.ndarray[tuple[()], np.dtype]:
    reachable = get_reachable_mask(grid, position)
    unknown = (grid == -1)
    enclosed = unknown & (~reachable)
    grid[enclosed] = VAL_INACCESSIBLE
    
    return grid

def fill_boundary_unknowns(grid: np.ndarray, position: tuple[int, int]) -> np.ndarray[tuple[()], np.dtype]:
    reachable = get_reachable_mask(grid, position)
    h, w = grid.shape
    for y in range(h):
        for x in range(w):
            if grid[y, x] == -1 and reachable[y, x]:
                neighbors = [(x+dx, y+dy) for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]]
                if any(0 <= nx < w and 0 <= ny < h and grid[ny, nx] == 0 for nx, ny in neighbors):
                    grid[y, x] = 0
    
    return grid

def fill_boundary_gaps(grid: np.ndarray, position: tuple[int, int]) -> np.ndarray[tuple[()], np.dtype]:
    reachable = get_reachable_mask(grid, position)
    h, w = grid.shape
    for y in range(h):
        for x in range(w):
            if grid[y, x] == -1 and reachable[y, x]:
                neighbors = [(x+dx, y+dy) for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]]
                free_neighbors = [grid[ny, nx] == 0 for nx, ny in neighbors if 0 <= nx < w and 0 <= ny < h]
                if sum(free_neighbors) >= 2:
                    grid[y, x] = 0
    
    return grid

In [8]:
def get_position(x: float, y: float, origin_x: float = 0.0, origin_y: float = 0.0) -> tuple[int, int]:
    """Get position in pixels from real-world coordinates.

    Args:
        x (float), y (float): Real-world coordinates to mark.
        origin_x (float, optional), origin_y (float, optional): Real-world coordinates of the map's origin (bottom-left corner). Defaults to 0.0.

    Returns:
        tuple: position x and y in pixels
    """
    
    return int((x - origin_x) / RESOLUTION), int((y - origin_y) / RESOLUTION)

def mark_position(
    grid: np.ndarray,
    x: float,
    y: float,
    origin_x: float = 0.0,
    origin_y: float = 0.0
    ) -> np.ndarray:
    """
    Marks a position on the grid based on real-world coordinates and map origin.

    Args:
        grid: 2D numpy array representing the map.
        x, y: Real-world coordinates to mark.
        origin_x, origin_y: Real-world coordinates of the map's origin (bottom-left corner).

    Returns:
        Modified grid with the position marked.
    """
    height, width = grid.shape

    # Convert world coordinates to pixel indices
    x_pixel, y_pixel = get_position(x, y, origin_x, origin_y)

    print(f"Marking position at world coordinates: ({x}, {y})")
    print(f"Converted to pixel coordinates: ({x_pixel}, {y_pixel})")

    if 0 <= x_pixel < width and 0 <= y_pixel < height:
        grid[y_pixel, x_pixel] = VAL_CURR_POSITION
    else:
        print("Warning: Position out of grid bounds.")

    return grid

def mark_position_v2(
    grid: np.ndarray,
    x: int,
    y: int,
    color: int = VAL_CURR_POSITION
    ) -> np.ndarray:
    """
    Marks a position on the grid based on real-world coordinates and map origin.

    Args:
        grid: 2D numpy array representing the map.
        x, y: grid coordinates.

    Returns:
        Modified grid with the position marked.
    """
    height, width = grid.shape

    print(f"Marking position at world coordinates: ({x}, {y})")

    if 0 <= x < width and 0 <= y < height:
        grid[y, x] = color
    else:
        print("Warning: Position out of grid bounds.")

    return grid

In [9]:
_map = read_data_map()
_odom = read_data_odom()

if _map is not None and _odom is not None:
    grid, origin_x, origin_y = _map
    position_x, position_y = _odom
    
    # Convert position to grid coordinates
    pos_x_grid = int((position_x - origin_x) / RESOLUTION)
    pos_y_grid = int((position_y - origin_y) / RESOLUTION)
    position = (pos_x_grid, pos_y_grid)

    filled_grid = fill_enclosed_unknowns_v2(grid, position)
    fully_enclosed = is_fully_enclosed(filled_grid, position)
    # print(fully_enclosed, _f)
    if fully_enclosed:
        filled_grid = fill_outside_with_val_inaccessible(filled_grid, position)

    # Mark current position on the grid
    grid[pos_y_grid, pos_x_grid] = VAL_CURR_POSITION

    # Find frontiers and perform greedy exploration
    path, grids = clustering_frontier_exploration(grid, (pos_y_grid, pos_x_grid))
    # path = ant_colony_exploration(grid, (pos_y_grid, pos_x_grid))
    print("Path found:", path)
    
    # Mark next goal on the grid
    for idx, next_goal in enumerate(path):
        grid = grids[idx]
        grid[next_goal[0], next_goal[1]] = VAL_NEXT_GOAL

        # Count values for legend
        occupied_count = np.sum(grid == VAL_OCCUPIED)
        free_count = np.sum(grid == VAL_FREE)
        inaccessible_count = np.sum(grid == VAL_INACCESSIBLE)
        unknown_count = np.sum(grid == VAL_UNKNOWN)
        
        explored_percent = (occupied_count + free_count) / grid.size

        print_plot_v2(
            grid,
            title="Explored Map",
            filename=MAP_001,
            occupied=occupied_count,
            free=free_count,
            inaccessible=inaccessible_count,
            unknown=unknown_count,
            explored_percent=explored_percent,
            position_x=position_x,
            position_y=position_y
        )
    

{'header': {'stamp': {'sec': 62, 'nanosec': 417000000}, 'frame_id': 'map'}, 'info': {'map_load_time': {'sec': 0, 'nanosec': 0}, 'resolution': 0.029999999329447746, 'width': 187, 'height': 174, 'origin': {'position': {'x': -2.9626397491207124, 'y': -2.6577183216049107, 'z': 0.0}, 'orientation': {'x': 0.0, 'y': 0.0, 'z': 0.0, 'w': 1.0}}}, 'data': [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 

ValueError: too many values to unpack (expected 2)